In [1]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split

## Compute Device Selection

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("using device:", device)

using device: mps


## Data Loading

In [3]:
X = np.load("train_embeddings.npy")
y = np.load("train_prices.npy")

In [4]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)
 
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                          torch.tensor(y_train, dtype=torch.float32))
val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                        torch.tensor(y_val, dtype=torch.float32))
 
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

In [5]:
baseline_pred = np.full_like(y_val, y_train.mean())
baseline_mse = np.mean((y_val - baseline_pred) ** 2)
print(f"predict-mean baseline val MSE: {baseline_mse:,.0f}")

predict-mean baseline val MSE: 158,531


## Model Definition

In [6]:
class PriceMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )
 
    def forward(self, x):
        return self.net(x).squeeze(-1)   # (B, 1) -> (B,)

In [7]:
model = PriceMLP(input_dim=X.shape[1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
loss_fn = nn.MSELoss()

In [8]:
n_epochs = 100
patience = 10
best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0
 
for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_ds)
 
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_loss += loss_fn(pred, yb).item() * len(xb)
    val_loss /= len(val_ds)
 
    print(f"epoch {epoch+1:3d}/{n_epochs}  train MSE {train_loss:,.0f}  val MSE {val_loss:,.0f}")
 
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break
 
model.load_state_dict(best_state)
r2 = 1 - best_val_loss / np.var(y_val)
print(f"\nbest val MSE: {best_val_loss:,.0f}   R2: {r2:.3f}   "
      f"(baseline MSE: {baseline_mse:,.0f}, baseline R2: 0.000)")
 
torch.save(model.state_dict(), "price_mlp.pth")
print("saved price_mlp.pth (best val checkpoint)")

epoch   1/100  train MSE 666,351  val MSE 627,553
epoch   2/100  train MSE 549,280  val MSE 428,980
epoch   3/100  train MSE 311,717  val MSE 216,343
epoch   4/100  train MSE 188,629  val MSE 174,772
epoch   5/100  train MSE 166,195  val MSE 156,409
epoch   6/100  train MSE 149,508  val MSE 140,074
epoch   7/100  train MSE 135,494  val MSE 127,318
epoch   8/100  train MSE 125,178  val MSE 119,111
epoch   9/100  train MSE 118,596  val MSE 113,905
epoch  10/100  train MSE 114,212  val MSE 111,719
epoch  11/100  train MSE 111,304  val MSE 108,423
epoch  12/100  train MSE 108,856  val MSE 106,881
epoch  13/100  train MSE 106,962  val MSE 105,374
epoch  14/100  train MSE 105,199  val MSE 104,232
epoch  15/100  train MSE 103,887  val MSE 104,026
epoch  16/100  train MSE 102,546  val MSE 102,939
epoch  17/100  train MSE 101,620  val MSE 102,256
epoch  18/100  train MSE 100,372  val MSE 102,448
epoch  19/100  train MSE 99,831  val MSE 100,911
epoch  20/100  train MSE 98,729  val MSE 100,372
ep

In [9]:
model.eval()
with torch.no_grad():
    val_pred = model(torch.tensor(X_val, dtype=torch.float32).to(device)).cpu().numpy()
 
abs_err = np.abs(y_val - val_pred)
pct_err = abs_err / y_val * 100
 
print(f"\nRMSE:  ${np.sqrt(np.mean((y_val - val_pred) ** 2)) * 1000:,.0f}")
print(f"MAE:   ${abs_err.mean() * 1000:,.0f}")
print(f"MedAE: ${np.median(abs_err) * 1000:,.0f}")
print(f"MAPE:  {pct_err.mean():.1f}%")
print(f"MdAPE: {np.median(pct_err):.1f}%")
 
torch.save(model.state_dict(), "price_mlp.pth")
print("saved price_mlp.pth (best val checkpoint)")


RMSE:  $304,193
MAE:   $223,205
MedAE: $165,264
MAPE:  36.1%
MdAPE: 26.5%
saved price_mlp.pth (best val checkpoint)


In [12]:
"""
Ridge regression on cached DINOv3 embeddings — same train/val split as train_mlp.py,
so R2 is directly comparable.

Expects:
    train_embeddings.npy   (N, embed_dim)
    train_prices.npy       (N,)
"""
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.pipeline import make_pipeline

X = np.load("train_embeddings.npy")
y = np.load("train_prices.npy")

# same split as train_mlp.py — same random_state, so results are apples-to-apples
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

baseline_mse = np.mean((y_val - y_train.mean()) ** 2)
print(f"predict-mean baseline val MSE: {baseline_mse:,.0f}")

model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(0, 5, 11)))
model.fit(X_train, y_train)
pred = model.predict(X_val)

mse = np.mean((y_val - pred) ** 2)
r2 = 1 - mse / np.var(y_val)
print(f"ridge val MSE: {mse:,.0f}   R2: {r2:.3f}   (alpha={model[-1].alpha_:.0f})")

pct_err = np.abs(y_val - pred) / y_val * 100
print(f"MAPE:  {pct_err.mean():.1f}%")
print(f"MdAPE: {np.median(pct_err):.1f}%")

predict-mean baseline val MSE: 158,531
ridge val MSE: 96,272   R2: 0.393   (alpha=1000)
MAPE:  37.9%
MdAPE: 27.4%


In [13]:
# LassoCV picks alpha via internal cross-validation on the training set only
model = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=10000, n_jobs=-1))
model.fit(X_train, y_train)
pred = model.predict(X_val)
 
mse = np.mean((y_val - pred) ** 2)
r2 = 1 - mse / np.var(y_val)
n_nonzero = np.sum(model[-1].coef_ != 0)
print(f"lasso val MSE: {mse:,.0f}   R2: {r2:.3f}   (alpha={model[-1].alpha_:.4f})")
print(f"nonzero coefficients: {n_nonzero} / {X.shape[1]}")
 
pct_err = np.abs(y_val - pred) / y_val * 100
print(f"MAPE:  {pct_err.mean():.1f}%")
print(f"MdAPE: {np.median(pct_err):.1f}%")

/opt/anaconda3/envs/COMP90086_PyTorch/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:825: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.220480e+05, tolerance: 7.946e+04
  model = cd_fast.enet_coordinate_descent_gram(


lasso val MSE: 96,648   R2: 0.390   (alpha=1.4821)
nonzero coefficients: 364 / 768
MAPE:  38.2%
MdAPE: 27.8%


In [14]:
coef = model[-1].coef_
half = len(coef) // 2   # embedding = [CLS | patch_mean], each of this width

cls_nonzero = np.sum(coef[:half] != 0)
patch_nonzero = np.sum(coef[half:] != 0)

print(f"CLS dims kept:        {cls_nonzero} / {half}")
print(f"patch-mean dims kept: {patch_nonzero} / {half}")

CLS dims kept:        230 / 384
patch-mean dims kept: 134 / 384
